In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from datasets import load_dataset, concatenate_datasets, load_from_disk

from pyreft import (
    TaskType,
    get_reft_model,
    ReftConfig,
    ReftTrainerForCausalLM, 
    ReftDataCollator,
    ReftSupervisedDataset,
    LoreftIntervention
)

prompt_no_input_template = """Below is an instruction that \
describes a task. Write a response that appropriately \
completes the request.

### Instruction:
%s

### Response:
"""
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
device = "cuda" if torch.cuda.is_available() else "cpu"


nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


In [3]:
# # load model (take 1 min)
model_name_or_path = "/data/chaojian/Llama-2-7b-hf" # yahma/llama-7b-hf or yahma/llama-13b-hf
# model = AutoModelForCausalLM.from_pretrained(
#      model_name_or_path, torch_dtype=torch.bfloat16, device_map=device)

# # get tokenizer
model_max_length = 512
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path, model_max_length=model_max_length, 
    padding_side="right", use_fast=False)
tokenizer.pad_token = tokenizer.unk_token

In [5]:
tokenizer.eos_token_id,tokenizer.bos_token_id

(2, 1)

In [6]:
tokenizer.bos_token

'<s>'

In [8]:
tokenizer(['i like deep learning </s>', 'i like machine learning </s>'], return_tensors='pt', padding=True)

{'input_ids': tensor([[    1,   474,   763,  6483,  6509, 29871,     2],
        [    1,   474,   763,  4933,  6509, 29871,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1]])}

In [3]:

class SubloreftIntervention(LoreftIntervention):
    """
    This is a LoReFT that supports subspace interventions!
    """
    def forward(
        self, base, source=None, subspaces=None
    ):
        assert subspaces is not None
        output = []

        # print("============== Subloreft ===============")
        # print("=== Debug SubloreftIntervention.forward ===")
        # print("base shape:", base.shape)
        # print("subspaces:", subspaces)
        # print("len(subspaces):", len(subspaces))
        # print("============== Subloreft ===============")
        
        rotated_base = self.rotate_layer(base)
        print("rotated_base shape:", rotated_base.shape)

        diff = self.act_fn(self.learned_source(base)) - rotated_base
        print("diff shape:", diff.shape)

        batched_subspace = []
        batched_weights = []
        
        for example_i in range(len(subspaces)):
            LHS = (diff[example_i, :, subspaces[example_i]])
            RHS = self.rotate_layer.weight[..., subspaces[example_i]].T
            # print(diff.shape, LHS.shape, RHS.shape, base.shape, subspaces)
            # 
            # torch.Size([5, 2, 8]) torch.Size([2, 4]) torch.Size([4, 4096]) torch.Size([5, 2, 4096]) [[4, 5, 6, 7], [4, 5, 6, 7], [4, 5, 6, 7], [4, 5, 6, 7], [4, 5, 6, 7]]
            # print(f"example {example_i}:")
            # print("  LHS shape:", LHS.shape)
            # print("  RHS shape:", RHS.shape)
            batched_subspace += [LHS]
            batched_weights += [RHS]
        

        batched_subspace = torch.stack(batched_subspace, dim=0)
        batched_weights = torch.stack(batched_weights, dim=0)
        output = base + torch.bmm(batched_subspace, batched_weights)

        return self.dropout(output.to(base.dtype))

In [4]:
TARGET_LAYER = [16, 18, 20, 22, 24, 26, 28, 30]

# get reft model
reft_config = ReftConfig(representations=[{
        "layer": target_layer, "component": "block_output",
        "intervention": SubloreftIntervention(
        embed_dim=model.config.hidden_size, low_rank_dimension=24)
    }
    for target_layer in TARGET_LAYER 
   ])
reft_model = get_reft_model(model, reft_config)
reft_model.print_trainable_parameters()

Intervention key: layer_16_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_18_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_20_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_22_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_24_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_26_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_28_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_30_comp_block_output_unit_pos_nunit_1#0
trainable intervention params: 1,573,056 || trainable model params: 0
model params: 6,738,415,616 || trainable%: 0.023344597449063018


In [5]:
reft_model.load_intervention('/data/chaojian/Multi-alignment/tmp/checkpoint-6210/intervenable_model', 
                             include_model=True)


/data/chaojian/anaconda3/envs/alignment/lib/python3.10/site-packages/pyvene/models/intervenable_base.py:1379: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_state_dict 

In [6]:
len(reft_model.interventions)

8

In [14]:
instructions = [
    "Please choose the correct answer to the question: Which statement best explains why photosynthesis is the foundation of most food webs?\n\nAnswer1: Sunlight is the source of energy for nearly all ecosystems. Answer2: Most ecosystems are found on land instead of in water. Answer3: Carbon dioxide is more available than other gases. Answer4: The producers in all ecosystems are plants.\n\nAnswer format: answer1/answer2/answer3/answer4",

]
       
tokenizer.padding_side = "left"
prompt = [prompt_no_input_template % instruction for instruction in instructions]
# prompt = instruction
prompt = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
base_unit_location = prompt["input_ids"].shape[-1] - 1
print(base_unit_location)

shift = prompt["attention_mask"].argmax(dim=1).unsqueeze(1)  # last position
print(shift)

l = 5

prefix = torch.arange(l).repeat(len(instructions), 1).to(device) + shift
print(prefix)

# 创建后三个数 [x-2, x-1, x]
suffix = torch.tensor([base_unit_location - i for i in range(l-1, -1, -1)]).repeat(len(instructions), 1).to(device)

print(suffix)


# 拼接得到目标张量
base_unit_location_batched = torch.cat([prefix, suffix], dim=1)

print(base_unit_location_batched)


_, reft_response = reft_model.generate(
    prompt, unit_locations={"sources->base": (None, [

            base_unit_location_batched.tolist()

            ]*len(reft_model.interventions)
        )
    },
    subspaces=[[[4,5,6,7]]*len(instructions)]*len(reft_model.interventions),
    intervene_on_prompt=True, max_new_tokens=32, do_sample=False, 
    no_repeat_ngram_size=5, repetition_penalty=1.2,
    eos_token_id=tokenizer.eos_token_id, early_stopping=True,temperature=0.1,)
            
print(tokenizer.batch_decode(reft_response, skip_special_tokens=True))

145
tensor([[0]], device='cuda:0')
tensor([[0, 1, 2, 3, 4]], device='cuda:0')
tensor([[141, 142, 143, 144, 145]], device='cuda:0')
tensor([[  0,   1,   2,   3,   4, 141, 142, 143, 144, 145]], device='cuda:0')
++++++++++++++++++++++++++++
{'input_ids': tensor([[    1, 13866,   338,   385, 15278,   393, 16612,   263,  3414, 29889,
         14350,   263,  2933,   393,  7128,  2486,  1614,  2167,   278,  2009,
         29889,    13,    13,  2277, 29937,  2799,  4080, 29901,    13, 12148,
          6755,   278,  1959,  1234,   304,   278,  1139, 29901,  8449,  3229,
          1900, 18568,  2020, 20612,   948, 26533,   338,   278, 22778,   310,
          1556,  9687,  1856, 29879, 29973,    13,    13, 22550, 29896, 29901,
          8991,  4366,   338,   278,  2752,   310,  5864,   363,  8886,   599,
           321,  3944,   973, 29879, 29889,   673, 29906, 29901,  7849,   321,
          3944,   973, 29879,   526,  1476,   373,  2982,  2012,   310,   297,
          4094, 29889,   673, 29941, 

/data/chaojian/anaconda3/envs/alignment/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/chaojian/anaconda3/envs/alignment/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/data/chaojian/anaconda3/envs/alignment/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:649: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


In [8]:
base_unit_location

tensor([112, 137], device='cuda:0')

In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name_or_path = "/data/chaojian/Llama-2-7b-hf" # yahma/llama-7b-hf or yahma/llama-13b-hf
model = AutoModelForCausalLM.from_pretrained(
     model_name_or_path, torch_dtype=torch.bfloat16, device_map='cuda:1')

# get tokenizer
model_max_length = 512
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path, model_max_length=model_max_length, 
    padding_side="right", use_fast=False)
tokenizer.pad_token = tokenizer.unk_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
prompt_no_input_template = """Below is an instruction that \
describes a task. Write a response that appropriately \
completes the request.

### Instruction:
%s

### Response:
"""
instruction = "Please choose the correct answer to the question: Which of the following statements best explains why magnets usually stick to a refrigerator door?\n\nAnswer1: The refrigerator door is smooth. Answer2: The refrigerator door contains iron. Answer3: The refrigerator door is a good conductor. Answer4: The refrigerator door has electric wires in it.\n\nAnswer format: answer1/answer2/answer3/answer4",
    
prompt = prompt_no_input_template % instruction
# prompt = instruction
prompt = tokenizer(prompt, return_tensors="pt").to('cuda:1')

In [8]:
output = model.generate(**prompt, max_new_tokens=5, do_sample=False, 
    no_repeat_ngram_size=5, repetition_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id, early_stopping=True, temperature=0.7)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Please choose the correct answer to the question: Which of the following statements best explains why magnets usually stick to a refrigerator door?

Answer1: The refrigerator door is smooth. Answer2: The refrigerator door contains iron. Answer3: The refrigerator door is a good conductor. Answer4: The refrigerator door has electric wires in it.

Answer format: answer1/answer2/answer3/answer4

### Response:

#### Answer 1


In [19]:
a.permute(1, 0, 2).tolist()

[[[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]]]

In [23]:
[[[0,1,2,3,4, base_unit_location-1,base_unit_location-1,base_unit_location-2,base_unit_location-1,base_unit_location]]]*len(reft_model.interventions)

[[[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]],
 [[0, 1, 2, 3, 4, 42, 42, 41, 42, 43]]]

In [25]:
b = torch.tensor([[[0, base_unit_location]], [[0, base_unit_location]]])


In [7]:
a = torch.tensor([[[1]],[[3]]])
a.shape

torch.Size([2, 1, 1])

In [17]:
b = torch.tensor(
[

            base_unit_location_batched.tolist()

            ]*len(reft_model.interventions)

)

b.shape

torch.Size([8, 2, 8])

In [16]:
[base_unit_location_batched.tolist()] * 3

[[[0, 1, 2, 3, 109, 110, 111, 112], [0, 1, 2, 3, 134, 135, 136, 137]],
 [[0, 1, 2, 3, 109, 110, 111, 112], [0, 1, 2, 3, 134, 135, 136, 137]],
 [[0, 1, 2, 3, 109, 110, 111, 112], [0, 1, 2, 3, 134, 135, 136, 137]]]

In [27]:
base_unit_location.unsqueeze(1)

tensor([[112],
        [137]], device='cuda:0')

In [40]:
location = torch.tensor([[112],
                         [137]])

# 创建前三个数 [0, 1, 2]
prefix = torch.arange(3).repeat(location.size(0), 1)

# 创建后三个数 [x-2, x-1, x]\
l = 3
suffix = torch.cat([location - i for i in range(l-1, -1, -1)], dim=1)

# 拼接得到目标张量
result = torch.cat([prefix, suffix], dim=1)

result

tensor([[  0,   1,   2, 110, 111, 112],
        [  0,   1,   2, 135, 136, 137]])

In [33]:
[result.tolist()]*3

[[[0, 1, 2, 110, 111, 112], [0, 1, 2, 135, 136, 137]],
 [[0, 1, 2, 110, 111, 112], [0, 1, 2, 135, 136, 137]],
 [[0, 1, 2, 110, 111, 112], [0, 1, 2, 135, 136, 137]]]

In [39]:
for i in range(2, -1, -1):
    print(i)

2
1
0


In [3]:
import torch
d = torch.tensor([[[4,5,6,7]]*3]*5)
d.shape

torch.Size([5, 3, 4])

In [13]:
"../Llama2-7b-hf/".lstrip("../").rstrip("/")

'Llama2-7b-hf'

In [3]:
import json

with open('/data/chaojian/Multi-alignment/multi_train/experiment/Llama-2-7b-hf-ARC-Easy.json', 'r') as f:
    data = json.load(f)

n = len(data)
correct = 0
for item in data:
    if item['flag'] == True:
        correct += 1

print(correct / n) 
    

0.7529461279461279
